# Tool Execution - Files, Timeouts, and Error Handling

## Purpose
Learn advanced tool execution patterns including returning file content, setting timeouts, and providing custom error messages. These patterns are essential for production systems that need robust tool behavior.

## Key Concepts
- **ToolOutputFileContent**: Return binary files (PDFs, images) from tools
- **ToolOutputText**: Return text explanations alongside files
- **timeout**: Set execution time limits for tools
- **failure_error_function**: Customize error messages for tool failures

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Part 1: File Outputs from Tools

Tools can return file content (PDFs, images, documents) that agents can read and process.

**Use Cases**:
- Load reports or charts for analysis
- Generate visualizations and return as images
- Fetch documents from storage

💡 **Key Pattern**: Return list with both `ToolOutputText` (explanation) and `ToolOutputFileContent` (file data).

### Import File Output Types

In [ ]:
import asyncio
import base64
from pathlib import Path

from agents import Agent, Runner, function_tool
from agents.tool import ToolOutputFileContent, ToolOutputText

### Step 1: Create Tool That Returns File

Define a tool that loads a file and returns it with explanation:

**File Output Structure**:
1. Read file as bytes
2. Base64 encode for transmission
3. Create data URI: `data:mime/type;base64,{encoded_data}`
4. Return list with `ToolOutputText` and `ToolOutputFileContent`

⚡ **Important**: Agent can read and understand file content (PDFs, images with vision models).

In [ ]:
@function_tool
def load_sales_report() -> list:
    """
    Load the monthly sales report PDF from disk.
    Returns the file itself plus a short note so the model knows what it got.
    """
    pdf_path = Path("sales_report.pdf")
    pdf_bytes = pdf_path.read_bytes()

    # Encode the PDF as a base64 data URI for file_data
    b64 = base64.b64encode(pdf_bytes).decode("utf-8")

    return [
        ToolOutputText(
            text="Loaded sales_report.pdf. Use the attached file to summarize "
            "total revenue and the top region."
        ),
        ToolOutputFileContent(
            file_data=f"data:application/pdf;base64,{b64}",
            filename="sales_report.pdf",
        ),
    ]

### Step 2: Create Agent That Processes Files

Agent receives file content and can analyze it:

In [ ]:
agent = Agent(
    name="Sales Analyst",
    model=model_id,
    instructions=(
        "You are a sales analyst. When asked for a report, call "
        "load_sales_report, then read the returned PDF file content and "
        "answer with: total revenue, the top region by revenue, and a one-line "
        "takeaway. Keep it concise."
    ),
    tools=[load_sales_report],
)

### Step 3: Run and Analyze File

🎯 **Result**: Agent reads PDF content and provides analysis!

In [ ]:
result = await Runner.run(agent, "Give me country wise sales breakdown.")
print(result.final_output)

## Part 2: Tool Timeouts

Set execution time limits to prevent tools from hanging indefinitely.

**Parameters**:
- `timeout`: Time limit in seconds (float)
- `timeout_behavior`: `"raise_exception"` or `"return_error_message"`

💡 **Use Case**: External API calls, database queries, file processing - any operation that might hang.

### Import Timeout Error

In [ ]:
from agents import ToolTimeoutError

### Step 4: Create Tool with Timeout

Define a tool that will exceed its timeout limit:

⚡ **Configuration**: `timeout=1.5` seconds, but tool takes 5 seconds → timeout triggers!

In [ ]:
@function_tool(timeout=1.5, timeout_behavior="raise_exception")
async def slow_tool() -> str:
    """A tool that takes too long to execute."""
    await asyncio.sleep(5)  # Will timeout before completing
    return "done"

### Step 5: Test Timeout Behavior

In [ ]:
agent_timeout = Agent(
    name="timeout-demo-agent",
    model=model_id,
    tools=[slow_tool]
)

### Step 6: Run and Observe Timeout

🔍 **Watch**: Tool execution is capped at 1.5 seconds; on timeout it raises an exception by design. This exception is expected, so don't halt on it — simply proceed to the next cell in the notebook

In [ ]:
result = await Runner.run(agent_timeout, "Run the tool")
print(result.final_output)

## Part 3: Custom Error Handling

Provide user-friendly error messages when tools fail, hiding technical details.

**Use Cases**:
- Convert technical errors to user-friendly messages
- Log errors for debugging while showing clean messages
- Provide actionable guidance when tools fail

💡 **Pattern**: `failure_error_function` receives exception, returns string shown to user.

### Import Context Types

In [ ]:
from agents import RunContextWrapper
from typing import Any

### Step 7: Define Custom Error Handler

Create function that transforms exceptions into user-friendly messages:

**Function Signature**:
- `context`: RunContextWrapper with execution state
- `error`: The exception that was raised
- Returns: User-facing error string

🔍 **Best Practice**: Log technical details, return friendly message.

In [ ]:
def my_custom_error_function(context: RunContextWrapper[Any], error: Exception) -> str:
    """A custom function to provide a user-friendly error message."""
    print(f"A tool call failed with the following error: {error}")
    return "An internal server error occurred. Please try again later"

### Step 8: Create Tool with Custom Error Handler

Apply error handler to tool using decorator parameter:

In [ ]:
@function_tool(failure_error_function=my_custom_error_function)
def get_user_profile(user_id: str) -> str:
    """Fetches a user profile."""
    if user_id == "user_123":
        return "User profile for user_123 successfully retrieved."
    else:
        raise ValueError(f"Could not retrieve profile for user_id: {user_id}. API returned an error.")

### Step 9: Test Error Handling

In [ ]:
agent_error = Agent(
    name="profile_agent",
    model=model_id,
    instructions="Retrieve profile of the individual users.",
    tools=[get_user_profile]
)

### Step 10: Trigger Error and See Custom Message

🎯 **Result**: Technical error logged, but user sees friendly message!

In [ ]:
result = await Runner.run(agent_error, "Get profile of the user_1234")
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Tool Execution** notebook!